# Notebook 4 — expanding ejecta and the Sobolev resonance (the E1 capstone)

Level-1 code: homologous flow $v = r/t$; the comoving frequency $\nu_{\rm com}(r) = \nu_{\rm lab}(1 - r/ct)$; the resonance radius $r_{\rm res}$ of every line; the toy Sobolev depth $\tau_S$; the interaction coin $P_{\rm int} = 1 - e^{-\tau_S}$; and, as the exercise, the whole sequence in about thirty lines compared with `rtedu.sobolev.resonance_crossings`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))   # rtedu, uninstalled (education/src)
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI, GROUP_COLOUR
rng = np.random.default_rng(rtedu.SEEDS["ch04"])

In [ ]:
C = rtedu.C; SIGMA = rtedu.SIGMA_CLASSICAL
from rtedu.sobolev import toy_line_list
L = toy_line_list()                          # one source for this notebook, the video and the tests
t = L["t"]                                   # the epoch: v = r/t (2 days)
v_max = L["v_max_c"] * C; r_out = L["r_out"] # the outer edge
nu_lab = L["nu_lab"]                         # a 500 nm packet, launched at r = 0 moving radially outward
frac, n_l, f_osc = L["frac"], L["n_l"], np.full(L["frac"].size, L["f_osc"])   # twelve toy lines below the launch frequency
nu_lines = frac * nu_lab
lambda0 = C / nu_lines                       # cm
tau_S = SIGMA * f_osc * n_l * lambda0 * t    # the toy Sobolev depth, by hand (production shape: sobolev/optical_depth.py::tau_sobolev)
assert np.allclose(tau_S, L["tau"])
p_int = 1.0 - np.exp(-tau_S)
beta = (1.0 - np.exp(-tau_S)) / tau_S        # the OTHER Sobolev quantity: the escape probability (used from chapter 6 on)
for k in range(frac.size):
    print(f"line {k:2d}: nu/nu_lab {frac[k]:.3f}  tau_S {tau_S[k]:6.2f}  P_int {p_int[k]:.3f}  beta {beta[k]:.3f}")

## The comoving frequency sweeps down

A packet of fixed lab frequency sees $\nu_{\rm com}(r) = \nu_{\rm lab}(1 - r/ct)$ (first order in $v/c$). It meets line $\ell$ where $\nu_{\rm com}(r_{\rm res}) = \nu_\ell$, i.e. at $r_{\rm res} = ct\,(1 - \nu_\ell/\nu_{\rm lab})$, in order of decreasing $\nu_\ell$, and only if $r_{\rm res} < r_{\rm out}$.

In [ ]:
r = np.linspace(0, r_out, 400)
nu_com = nu_lab * (1.0 - r / (C * t))
r_res = C * t * (1.0 - nu_lines / nu_lab)
reachable = r_res < r_out
print("lines reachable before the edge:", np.flatnonzero(reachable).tolist(), f"(nu_com at the edge = {nu_com[-1] / nu_lab:.3f} nu_lab)")

## Exercise (the E1 exit criterion): the whole sequence from scratch

Given a packet and the line list, find the next resonance, compute its $\tau_S$, flip the coin, continue or stop. Then compare with `rtedu.sobolev.resonance_crossings` **on the same random stream**.

In [ ]:
def find_next_interaction(rng, nu_lab, nu_lines, tau_lines, r0, r_out, t):
    """Walk one packet outward. Returns the list of crossings met, each
    (line, r_res, tau, P_int, interacted); the walk stops at the first hit."""
    nu_start = nu_lab * (1.0 - r0 / (C * t))               # comoving frequency where the packet is now
    r_res = C * t * (1.0 - nu_lines / nu_lab)                # where nu_com(r) equals each line
    reachable = (nu_lines < nu_start) & (r_res < r_out)      # below the packet now, and before the edge
    order = np.argsort(-nu_lines[reachable])                 # met in decreasing frequency = increasing r
    crossings = []
    for k in np.flatnonzero(reachable)[order]:
        P = 1.0 - np.exp(-tau_lines[k])                      # the interaction coin
        hit = rng.random() < P
        crossings.append((int(k), float(r_res[k]), float(tau_lines[k]), float(P), bool(hit)))
        if hit:
            break                                            # E1 stops here: what happens next is E2
    return crossings

from rtedu.sobolev import resonance_crossings, first_interaction
seed = rtedu.SEEDS["ch04"] + 1
mine = find_next_interaction(np.random.default_rng(seed), nu_lab, nu_lines, tau_S, 0.0, r_out, t)
ref = resonance_crossings(nu_lab, nu_lines, tau_S, 0.0, r_out, t, rng=np.random.default_rng(seed))
assert [m[0] for m in mine] == [c["line"] for c in ref] and [m[4] for m in mine] == [c["interacted"] for c in ref]
assert np.allclose([m[1] for m in mine], [c["r_res"] for c in ref])
print("crossings (line, r_res/r_out, tau, P_int, interacted):")
for m in mine:
    print(f"  {m[0]:2d}  {m[1] / r_out:.3f}  {m[2]:5.2f}  {m[3]:.3f}  {m[4]}")

## Many packets: the escape probability

The chance that a packet crosses all reachable lines without interacting is $\prod_\ell e^{-\tau_\ell} = e^{-\sum_\ell \tau_\ell}$.

In [ ]:
n_packets = 20_000
escaped = sum(first_interaction(nu_lab, nu_lines, tau_S, 0.0, r_out, t, rng) is None for _ in range(n_packets)) / n_packets
p_escape_exact = float(np.exp(-tau_S[reachable].sum()))
sigma = np.sqrt(p_escape_exact * (1 - p_escape_exact) / n_packets)
print(f"escaped {escaped:.4f}; exp(-sum tau) = {p_escape_exact:.4f}; binomial sigma {sigma:.4f}")
assert abs(escaped - p_escape_exact) < 4 * sigma
# where do the interactions happen? the first-hit line distribution
hits = np.zeros(frac.size, int)
for _ in range(n_packets):
    h = first_interaction(nu_lab, nu_lines, tau_S, 0.0, r_out, t, rng)
    if h is not None:
        hits[h["line"]] += 1
first_hit_fraction = hits / n_packets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].plot(r / r_out, nu_com / nu_lab, color=OI["blue"], lw=1.5, label=r"$\nu_{\rm com}(r)/\nu_{\rm lab}$")
for k in range(frac.size):
    axes[0].axhline(frac[k], color=OI["red"] if reachable[k] else "grey", lw=0.5 + 1.5 * min(tau_S[k], 3) / 3, alpha=0.8)
    if reachable[k]:
        axes[0].plot([r_res[k] / r_out], [frac[k]], "o", color=OI["red"], ms=4)
axes[0].set_xlabel(r"$r/r_{\rm out}$"); axes[0].set_ylabel(r"$\nu/\nu_{\rm lab}$"); axes[0].set_title("the sweep: lines (grey/red) met at the dots", fontsize=9); axes[0].legend(fontsize=8)
tt = np.geomspace(1e-2, 30, 200)
axes[1].loglog(tt, 1 - np.exp(-tt), color=OI["red"], label=r"$P_{\rm int} = 1 - e^{-\tau}$")
axes[1].loglog(tt, (1 - np.exp(-tt)) / tt, color=OI["green"], label=r"$\beta = (1 - e^{-\tau})/\tau$")
axes[1].loglog(tt, 1 / tt, ":", color="grey", label=r"$1/\tau$"); axes[1].set_ylim(1e-2, 1.5); axes[1].set_xlabel(r"$\tau_S$"); axes[1].legend(fontsize=8); axes[1].set_title("two different quantities called Sobolev", fontsize=9)
axes[2].bar(np.arange(frac.size), first_hit_fraction, color=OI["blue"]); axes[2].set_xlabel("line index (met in this order)"); axes[2].set_ylabel("fraction of packets stopped here")
axes[2].set_title(f"first interaction; escaped {escaped:.3f}", fontsize=9)
fig.tight_layout(); save_fig(fig, "ch04_sobolev_sweep")

In [ ]:
results.record("ch04", dict(t_days=t / rtedu.DAY, v_max_c=v_max / C, nu_lab=nu_lab, lambda_lab_nm=1e7 * C / nu_lab,
                            n_lines=int(frac.size), n_reachable=int(reachable.sum()), frac=frac, tau_S=tau_S, p_int=p_int, beta=beta,
                            r_res_over_r_out=r_res / r_out, reachable=reachable.tolist(), tau_sum_reachable=float(tau_S[reachable].sum()),
                            one_packet=[dict(line=m[0], r_res_over_r_out=m[1] / r_out, tau=m[2], p_int=m[3], interacted=m[4]) for m in mine],
                            n_packets=n_packets, escaped=escaped, p_escape_exact=p_escape_exact, sigma_escape=float(sigma),
                            first_hit_fraction=first_hit_fraction, nu_com_edge_over_lab=float(nu_com[-1] / nu_lab)))